# Análisis exploratorio — Discurso de género (Trump, 2025+)

## ⚠️ Aviso ético y de datos

Este notebook utiliza **datos sintéticos de muestra** únicamente para validar la infraestructura de análisis. **No son discursos reales de Donald Trump.** Reemplázalos por datos extraídos de fuentes oficiales (Casa Blanca, Remarks, proclamas, mensajes) una vez completada la Fase 2 de extracción.

El flujo de código reproduce la arquitectura del proyecto:
1. Carga de documentos con metadatos
2. Limpieza y normalización
3. Segmentación en unidades de análisis
4. Detección temática por diccionario configurable
5. Análisis cuantitativo y visualización
6. Exportación de resultados

In [ ]:
import json
import re
import hashlib
import uuid
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

## 1. Carga de datos de muestra

Documentos sintéticos con metadatos mínimos requeridos por el modelo de datos del proyecto.

In [ ]:
SAMPLE_DOCS = [
    {
        'document_id': str(uuid.uuid4()),
        'title': 'Remarks on Family Values and Protection of Children (sample)',
        'speaker': 'Donald J. Trump',
        'document_type': 'remarks',
        'date': '2025-02-10',
        'url': 'https://example.whitehouse.gov/remarks/sample-1',
        'attribution_type': 'verbatim_speaker',
        'text': (
            'My administration is committed to protecting children and supporting the family. ',
            'We believe in mothers and fathers raising sons and daughters in a strong household. ',
            'The traditional family is the foundation of our nation. We reject the radical woke agenda ',
            'that attacks these values. Our policy defends biological sex and common sense. ',
            'We support faith, freedom, and the protection of life.'
        ),
    },
    {
        'document_id': str(uuid.uuid4()),
        'title': 'Speech on Sports and Single-Sex Spaces (sample)',
        'speaker': 'Donald J. Trump',
        'document_type': 'speech',
        'date': '2025-03-05',
        'url': 'https://example.whitehouse.gov/speeches/sample-2',
        'attribution_type': 'verbatim_speaker',
        'text': (
            'We will protect women\'s sports and keep men out of women\'s locker rooms. ',
            'Biological reality matters. Girls should compete against girls. Boys against boys. ',
            'This is about fairness, safety, and common sense. The left wants to erase these distinctions, ',
            'but we stand with parents and families.'
        ),
    },
    {
        'document_id': str(uuid.uuid4()),
        'title': 'Message on Religion and Morality (sample)',
        'speaker': 'Donald J. Trump',
        'document_type': 'message',
        'date': '2025-03-20',
        'url': 'https://example.whitehouse.gov/messages/sample-3',
        'attribution_type': 'verbatim_speaker',
        'text': (
            'Our nation was built on faith and tradition. We defend religious liberty and traditional values. ',
            'Morality guides our laws. We reject the elite media and their attacks on Christianity. ',
            'God bless the American family.'
        ),
    },
    {
        'document_id': str(uuid.uuid4()),
        'title': 'Executive Order on Diversity and Equity Programs (sample)',
        'speaker': 'Donald J. Trump',
        'document_type': 'executive_order',
        'date': '2025-04-02',
        'url': 'https://example.whitehouse.gov/executive-orders/sample-4',
        'attribution_type': 'verbatim_speaker',
        'text': (
            'We are ending DEI programs in the federal government. No more divisive equity ideology. ',
            'Equality means equal opportunity under the law, not quotas. We reject feminism that attacks men ',
            'and the family structure. Biology is not a social construct.'
        ),
    },
]

df = pd.DataFrame(SAMPLE_DOCS)
df['date'] = pd.to_datetime(df['date'])
df['raw_hash'] = df['text'].apply(lambda t: hashlib.sha256(t.encode('utf-8')).hexdigest())
df['word_count'] = df['text'].apply(lambda t: len(t.split()))

print(df[['document_id', 'title', 'document_type', 'date', 'word_count', 'raw_hash']].to_string(index=False))

## 2. Limpieza y normalización

Elimina etiquetas HTML, normaliza espacios en blanco y calcula métricas básicas por documento.

In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r'<[^>]+>', ' ', text)          # quitar etiquetas HTML
    text = re.sub(r'\s+', ' ', text)             # normalizar saltos de línea y espacios
    text = text.strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df['clean_word_count'] = df['clean_text'].apply(lambda t: len(t.split()))
df['sentence_count'] = df['clean_text'].apply(lambda t: len([s for s in re.split(r'[.!?]+', t) if s.strip()]))
df['paragraph_count'] = df['clean_text'].apply(lambda t: len([p for p in t.split('\n') if p.strip()]))

print(df[['document_id', 'clean_word_count', 'sentence_count', 'paragraph_count']].to_string(index=False))

## 3. Segmentación en unidades de análisis

Ventana de 50 a 200 palabras por unidad, con identificador determinístico y orden dentro del documento.

In [ ]:
def segment_text(text: str, min_words: int = 50, max_words: int = 200):
    words = text.split()
    units = []
    i = 0
    idx = 0
    while i < len(words):
        chunk = words[i:i + max_words]
        if len(chunk) < min_words and i + max_words < len(words):
            chunk = words[i:i + max_words * 2]
        units.append({
            'unit_id': str(uuid.uuid4()),
            'unit_index': idx,
            'text': ' '.join(chunk),
            'word_count': len(chunk),
        })
        i += max_words
        idx += 1
    return units

all_units = []
for _, row in df.iterrows():
    for unit in segment_text(row['clean_text']):
        unit['document_id'] = row['document_id']
        unit['document_type'] = row['document_type']
        unit['date'] = row['date']
        unit['speaker'] = row['speaker']
        all_units.append(unit)

units_df = pd.DataFrame(all_units)
print(f'Unidades generadas: {len(units_df)}')
print(units_df[['unit_id', 'document_id', 'unit_index', 'word_count']].head(10).to_string(index=False))

## 4. Detección temática por diccionario configurable

Diccionario temático inicial en inglés, organizado por los temas del proyecto Trump/género.

In [ ]:
GENDER_DICTIONARY = {
    'genero_y_sexo': ['woman', 'women', 'man', 'men', 'male', 'female', 'gender', 'sex'],
    'familia': ['family', 'families', 'father', 'mother', 'parent', 'parents'],
    'infancia': ['child', 'children', 'childhood'],
    'cuerpo': ['body', 'pregnancy', 'pregnant', 'reproduction'],
    'feminismo': ['feminism', 'feminist', 'equality', 'equity', 'DEI'],
    'religion': ['religion', 'faith', 'god', 'morality', 'values', 'tradition'],
    'adversarios': ['woke', 'democrat', 'democrats', 'left', 'elite', 'media', 'activist'],
    'legitimacion': ['biology', 'biological', 'common sense', 'rights', 'freedom', 'protection'],
}

def detect_mentions(units_df: pd.DataFrame, dictionary: dict) -> pd.DataFrame:
    mentions = []
    for _, unit in units_df.iterrows():
        text_lower = unit['text'].lower()
        for theme, terms in dictionary.items():
            for term in terms:
                if term.lower() in text_lower:
                    mentions.append({
                        'unit_id': unit['unit_id'],
                        'document_id': unit['document_id'],
                        'theme': theme,
                        'term': term,
                        'match_type': 'exact',
                    })
    return pd.DataFrame(mentions)

mentions_df = detect_mentions(units_df, GENDER_DICTIONARY)
print(f'Menciones detectadas: {len(mentions_df)}')
print(mentions_df.head(10).to_string(index=False))

## 5. Análisis cuantitativo y visualización

Frecuencia de menciones por tema, densidad por cada 1.000 palabras y evolución temporal. Solo indicadores descriptivos, sin inferencia causal.

In [ ]:
# Frecuencia absoluta por tema
theme_counts = (
    mentions_df.groupby('theme')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

fig, ax = plt.subplots()
sns.barplot(data=theme_counts, x='count', y='theme', ax=ax, palette='viridis')
ax.set_title('Frecuencia de menciones temáticas por diccionario (muestra sintética)')
ax.set_xlabel('Número de menciones')
ax.set_ylabel('Tema')
plt.tight_layout()
plt.show()

print(theme_counts.to_string(index=False))

In [ ]:
# Densidad por 1.000 palabras por tema
total_words = units_df['word_count'].sum()
density = (
    mentions_df.groupby('theme')
    .size()
    .reset_index(name='mentions')
)
density['density_per_1k_words'] = (density['mentions'] / total_words) * 1000
density = density.sort_values('density_per_1k_words', ascending=False)

fig, ax = plt.subplots()
sns.barplot(data=density, x='density_per_1k_words', y='theme', ax=ax, palette='mako')
ax.set_title('Densidad de menciones por 1.000 palabras (muestra sintética)')
ax.set_xlabel('Menciones por 1.000 palabras')
ax.set_ylabel('Tema')
plt.tight_layout()
plt.show()

print(density.to_string(index=False))

In [ ]:
# Evolución temporal de menciones por fecha
mentions_with_date = mentions_df.merge(
    units_df[['unit_id', 'date']], on='unit_id', how='left'
)
timeline = (
    mentions_with_date.groupby('date')
    .size()
    .reset_index(name='mentions')
    .sort_values('date')
)

fig, ax = plt.subplots()
sns.lineplot(data=timeline, x='date', y='mentions', marker='o', ax=ax)
ax.set_title('Evolución temporal de menciones temáticas (muestra sintética)')
ax.set_xlabel('Fecha')
ax.set_ylabel('Menciones')
plt.tight_layout()
plt.show()

print(timeline.to_string(index=False))

## 6. Notas analíticas e interpretación

- Las gráficas anteriores son **indicadores descriptivos** sobre una muestra sintética.
- Una coincidencia léxica **no implica** automáticamente un significado político o ideológico.
- Toda codificación definitiva requiere revisión humana y un libro de códigos versionado (ver Fase 6 del plan).
- No se extraen conclusiones causales sin evidencia codificada y auditada.

In [ ]:
# Exportación de resultados intermedios
output_dir = Path('outputs/analisis')
output_dir.mkdir(parents=True, exist_ok=True)

mentions_df.to_csv(output_dir / 'mentions_sample.csv', index=False, encoding='utf-8')
units_df.to_parquet(output_dir / 'units_sample.parquet', index=False)
df.to_parquet(output_dir / 'corpus_sample.parquet', index=False)

print(f'Archivos exportados en: {output_dir.resolve()}')

## 7. Siguientes pasos

1. Reemplazar `SAMPLE_DOCS` por ingesta real desde `config/sources/` y la CLI de extracción.
2. Conectar este notebook al corpus maestro en `data/processed/corpus.parquet`.
3. Implementar codificación cualitativa asistida con libro de códigos en `config/codebooks/`.
4. Generar el dashboard Streamlit para exploración conjunta del corpus y las citas.